In [ ]:
# Task 3: MNIST Handwritten Digit Classification - Enhanced Proof of Concept (PoC)
# CSE424 - Machine Learning
# Enhanced version with deployment-ready features and comprehensive analysis

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
import json
import pickle
from datetime import datetime
import os

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("=" * 60)
print("MNIST DIGIT CLASSIFICATION - ENHANCED PROOF OF CONCEPT")
print("=" * 60)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Building production-ready ML pipeline with EDA insights\n")

# ==========================================
# SECTION 1: ENHANCED DATA LOADING & PREPROCESSING
# ==========================================

class MNISTProcessor:
    """Enhanced MNIST data processor with EDA insights"""

    def __init__(self):
        self.scaler = StandardScaler()
        self.important_pixels = None
        self.stats = {}

    def load_data(self):
        """Load and prepare MNIST dataset"""
        print("📂 LOADING DATASET...")

        # Download dataset
        path = kagglehub.dataset_download("oddrationale/mnist-in-csv")

        # Load data
        train_path = f"{path}/mnist_train.csv"
        test_path = f"{path}/mnist_test.csv"

        df_train = pd.read_csv(train_path)
        df_test = pd.read_csv(test_path)

        # Store dataset info
        self.stats['train_samples'] = len(df_train)
        self.stats['test_samples'] = len(df_test)
        self.stats['features'] = df_train.shape[1] - 1  # excluding label

        print(f"✅ Training samples: {self.stats['train_samples']:,}")
        print(f"✅ Test samples: {self.stats['test_samples']:,}")
        print(f"✅ Features per sample: {self.stats['features']}")

        return df_train, df_test

    def preprocess_data(self, df_train, df_test):
        """Enhanced preprocessing with EDA insights"""
        print("\n🔧 ENHANCED PREPROCESSING...")

        # Separate features and labels
        X_train_full = df_train.drop('label', axis=1)
        y_train_full = df_train['label']
        X_test = df_test.drop('label', axis=1)
        y_test = df_test['label']

        # Normalize pixel values (0-255 to 0-1)
        X_train_full = X_train_full.astype('float32') / 255.0
        X_test = X_test.astype('float32') / 255.0

        # Create train/validation split
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.2,
            random_state=42,
            stratify=y_train_full
        )

        # EDA-informed feature selection (high-variance pixels)
        pixel_variances = X_train.var()
        variance_threshold = np.percentile(pixel_variances, 75)  # Top 25% pixels
        self.important_pixels = pixel_variances > variance_threshold

        # Apply feature selection
        X_train_selected = X_train.loc[:, self.important_pixels]
        X_val_selected = X_val.loc[:, self.important_pixels]
        X_test_selected = X_test.loc[:, self.important_pixels]

        # Store preprocessing stats
        self.stats['selected_features'] = self.important_pixels.sum()
        self.stats['feature_reduction'] = (1 - self.stats['selected_features'] / self.stats['features']) * 100

        print(f"✅ Selected {self.stats['selected_features']} important features")
        print(f"✅ Feature reduction: {self.stats['feature_reduction']:.1f}%")

        return {
            'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
            'X_train_selected': X_train_selected, 'X_val_selected': X_val_selected,
            'X_test_selected': X_test_selected,
            'y_train': y_train, 'y_val': y_val, 'y_test': y_test
        }

# ==========================================
# SECTION 2: ENHANCED MODEL IMPLEMENTATIONS
# ==========================================

class EnhancedRandomForest:
    """Enhanced Random Forest with hyperparameter optimization"""

    def __init__(self):
        self.model = RandomForestClassifier(
            n_estimators=200,  # Increased from 100
            max_depth=25,      # Optimized depth
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
        self.training_time = None

    def train(self, X_train, y_train):
        """Train the enhanced Random Forest model"""
        print("🌲 TRAINING ENHANCED RANDOM FOREST...")
        start_time = datetime.now()

        self.model.fit(X_train, y_train)

        self.training_time = (datetime.now() - start_time).total_seconds()
        print(f"✅ Training completed in {self.training_time:.2f} seconds")

    def predict(self, X_test):
        """Make predictions"""
        return self.model.predict(X_test)

    def get_feature_importance(self):
        """Get feature importance scores"""
        return self.model.feature_importances_

class EnhancedCNN:
    """Enhanced CNN with advanced architecture and callbacks"""

    def __init__(self):
        self.model = None
        self.history = None
        self.training_time = None

    def create_model(self, input_shape=(28, 28, 1)):
        """Create enhanced CNN architecture"""
        model = keras.Sequential([
            # First convolutional block
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            layers.BatchNormalization(),
            layers.Conv2D(32, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Second convolutional block
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Third convolutional block
            layers.Conv2D(128, (3, 3), activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.25),

            # Dense layers
            layers.Flatten(),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(10, activation='softmax')
        ])

        # Enhanced compilation
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy',  # No need for one-hot encoding
            metrics=['accuracy']
        )

        self.model = model
        return model

    def train(self, X_train, y_train, X_val, y_val, epochs=20):
        """Train the enhanced CNN model with callbacks"""
        print("🧠 TRAINING ENHANCED CNN...")

        # Advanced callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-7,
                verbose=1
            )
        ]

        start_time = datetime.now()

        # Train model
        self.history = self.model.fit(
            X_train, y_train,
            batch_size=128,
            epochs=epochs,
            validation_data=(X_val, y_val),
            callbacks=callbacks,
            verbose=1
        )

        self.training_time = (datetime.now() - start_time).total_seconds()
        print(f"✅ Training completed in {self.training_time:.2f} seconds")

    def predict(self, X_test):
        """Make predictions"""
        predictions = self.model.predict(X_test)
        return np.argmax(predictions, axis=1)

    def predict_proba(self, X_test):
        """Get prediction probabilities"""
        return self.model.predict(X_test)

# ==========================================
# SECTION 3: COMPREHENSIVE EVALUATION SYSTEM
# ==========================================

class ModelEvaluator:
    """Comprehensive model evaluation and comparison"""

    def __init__(self):
        self.results = {}

    def evaluate_model(self, model_name, y_true, y_pred, training_time=None):
        """Comprehensive model evaluation"""
        print(f"\n📊 EVALUATING {model_name.upper()}...")

        # Basic metrics
        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

        # Store results
        self.results[model_name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'training_time': training_time,
            'predictions': y_pred
        }

        print(f"✅ Accuracy: {accuracy:.4f}")
        print(f"✅ Precision: {precision:.4f}")
        print(f"✅ Recall: {recall:.4f}")
        print(f"✅ F1-Score: {f1:.4f}")
        if training_time:
            print(f"✅ Training Time: {training_time:.2f}s")

        return self.results[model_name]

    def compare_models(self):
        """Compare all evaluated models"""
        print("\n🏆 MODEL COMPARISON SUMMARY")
        print("-" * 60)

        comparison_df = pd.DataFrame(self.results).T
        comparison_df = comparison_df.round(4)

        # Find best model for each metric
        best_accuracy = comparison_df['accuracy'].idxmax()
        best_f1 = comparison_df['f1_score'].idxmax()

        print(f"🥇 Best Accuracy: {best_accuracy} ({comparison_df.loc[best_accuracy, 'accuracy']:.4f})")
        print(f"🥇 Best F1-Score: {best_f1} ({comparison_df.loc[best_f1, 'f1_score']:.4f})")

        return comparison_df

# ==========================================
# SECTION 4: ADVANCED VISUALIZATION SYSTEM
# ==========================================

class AdvancedVisualizer:
    """Enhanced visualization for model analysis"""

    def __init__(self):
        plt.style.use('default')
        sns.set_palette("husl")

    def plot_training_history(self, history, model_name):
        """Plot enhanced training history"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'{model_name} Training History', fontsize=16, fontweight='bold')

        # Accuracy
        axes[0, 0].plot(history.history['accuracy'], label='Training', linewidth=2)
        axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0, 0].set_title('Model Accuracy')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # Loss
        axes[0, 1].plot(history.history['loss'], label='Training', linewidth=2)
        axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
        axes[0, 1].set_title('Model Loss')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

        # Top-2 Accuracy
        if 'top_2_accuracy' in history.history:
            axes[1, 0].plot(history.history['top_2_accuracy'], label='Training', linewidth=2)
            axes[1, 0].plot(history.history['val_top_2_accuracy'], label='Validation', linewidth=2)
            axes[1, 0].set_title('Top-2 Accuracy')
            axes[1, 0].set_xlabel('Epoch')
            axes[1, 0].set_ylabel('Top-2 Accuracy')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)

        # Learning Rate (if available)
        if 'lr' in history.history:
            axes[1, 1].plot(history.history['lr'], linewidth=2, color='red')
            axes[1, 1].set_title('Learning Rate Schedule')
            axes[1, 1].set_xlabel('Epoch')
            axes[1, 1].set_ylabel('Learning Rate')
            axes[1, 1].set_yscale('log')
            axes[1, 1].grid(True, alpha=0.3)
        else:
            # Performance summary
            final_acc = history.history['val_accuracy'][-1]
            final_loss = history.history['val_loss'][-1]
            epochs_trained = len(history.history['accuracy'])

            axes[1, 1].text(0.1, 0.7, f'Final Validation Accuracy: {final_acc:.4f}',
                           transform=axes[1, 1].transAxes, fontsize=12, fontweight='bold')
            axes[1, 1].text(0.1, 0.5, f'Final Validation Loss: {final_loss:.4f}',
                           transform=axes[1, 1].transAxes, fontsize=12)
            axes[1, 1].text(0.1, 0.3, f'Epochs Trained: {epochs_trained}',
                           transform=axes[1, 1].transAxes, fontsize=12)
            axes[1, 1].set_title('Training Summary')
            axes[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

    def plot_confusion_matrices(self, y_true, predictions_dict):
        """Plot enhanced confusion matrices"""
        n_models = len(predictions_dict)
        fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
        if n_models == 1:
            axes = [axes]

        for idx, (model_name, y_pred) in enumerate(predictions_dict.items()):
            cm = confusion_matrix(y_true, y_pred)

            # Calculate per-class accuracy
            per_class_acc = cm.diagonal() / cm.sum(axis=1)

            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                       cbar_kws={'label': 'Count'})
            axes[idx].set_title(f'{model_name} Confusion Matrix\nMean Per-Class Acc: {per_class_acc.mean():.3f}')
            axes[idx].set_xlabel('Predicted Label')
            axes[idx].set_ylabel('True Label')

        plt.tight_layout()
        plt.show()

    def plot_model_comparison(self, results_df):
        """Plot comprehensive model comparison"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Comprehensive Model Comparison', fontsize=16, fontweight='bold')

        metrics = ['accuracy', 'precision', 'recall', 'f1_score']
        titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

        for idx, (metric, title) in enumerate(zip(metrics, titles)):
            row, col = idx // 2, idx % 2

            bars = axes[row, col].bar(results_df.index, results_df[metric],
                                     color=['skyblue', 'lightcoral', 'lightgreen'][:len(results_df)],
                                     edgecolor='black', alpha=0.7)

            axes[row, col].set_title(f'{title} Comparison')
            axes[row, col].set_ylabel(title)
            axes[row, col].set_ylim(0, 1)
            axes[row, col].grid(True, alpha=0.3)

            # Add value labels on bars
            for bar, value in zip(bars, results_df[metric]):
                axes[row, col].text(bar.get_x() + bar.get_width()/2, value + 0.01,
                                   f'{value:.4f}', ha='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

# ==========================================
# SECTION 5: DEPLOYMENT-READY FEATURES
# ==========================================

class DigitClassifier:
    """Production-ready digit classifier"""

    def __init__(self, model, preprocessor):
        self.model = model
        self.preprocessor = preprocessor

    def predict_single_digit(self, image_array):
        """Predict a single digit with confidence scores"""
        # Ensure proper shape and normalization
        if image_array.max() > 1:
            image_array = image_array.astype('float32') / 255.0

        if len(image_array.shape) == 2:
            image_array = image_array.reshape(1, 28, 28, 1)
        elif len(image_array.shape) == 3:
            image_array = image_array.reshape(1, *image_array.shape)

        # Get prediction probabilities
        if hasattr(self.model, 'predict_proba'):
            probabilities = self.model.predict_proba(image_array)[0]
        else:
            probabilities = self.model.predict(image_array)[0]

        predicted_digit = np.argmax(probabilities)
        confidence = probabilities[predicted_digit]

        return {
            'predicted_digit': int(predicted_digit),
            'confidence': float(confidence),
            'all_probabilities': probabilities.tolist()
        }

    def save_model(self, filepath):
        """Save model for deployment"""
        if hasattr(self.model, 'save'):
            self.model.save(filepath)
        else:
            with open(filepath, 'wb') as f:
                pickle.dump(self.model, f)
        print(f"✅ Model saved to {filepath}")

# ==========================================
# MAIN EXECUTION PIPELINE
# ==========================================

def main():
    """Main execution pipeline for enhanced PoC"""

    # Initialize components
    processor = MNISTProcessor()
    evaluator = ModelEvaluator()
    visualizer = AdvancedVisualizer()

    # Step 1: Load and preprocess data
    df_train, df_test = processor.load_data()
    data = processor.preprocess_data(df_train, df_test)

    # Step 2: Train Enhanced Random Forest
    print("\n" + "="*60)
    print("PHASE 1: ENHANCED RANDOM FOREST")
    print("="*60)

    rf_model = EnhancedRandomForest()
    rf_model.train(data['X_train_selected'], data['y_train'])
    rf_predictions = rf_model.predict(data['X_test_selected'])

    # Evaluate Random Forest
    rf_results = evaluator.evaluate_model('Enhanced_Random_Forest',
                                         data['y_test'], rf_predictions,
                                         rf_model.training_time)

    # Step 3: Train Enhanced CNN
    print("\n" + "="*60)
    print("PHASE 2: ENHANCED CNN")
    print("="*60)

    cnn_model = EnhancedCNN()
    cnn_model.create_model()

    # Reshape data for CNN
    X_train_cnn = data['X_train'].values.reshape(-1, 28, 28, 1)
    X_val_cnn = data['X_val'].values.reshape(-1, 28, 28, 1)
    X_test_cnn = data['X_test'].values.reshape(-1, 28, 28, 1)

    cnn_model.train(X_train_cnn, data['y_train'], X_val_cnn, data['y_val'], epochs=15)
    cnn_predictions = cnn_model.predict(X_test_cnn)

    # Evaluate CNN
    cnn_results = evaluator.evaluate_model('Enhanced_CNN',
                                          data['y_test'], cnn_predictions,
                                          cnn_model.training_time)

    # Step 4: Comprehensive Analysis
    print("\n" + "="*60)
    print("PHASE 3: COMPREHENSIVE ANALYSIS")
    print("="*60)

    # Model comparison
    comparison_df = evaluator.compare_models()
    print("\n📊 Detailed Comparison:")
    print(comparison_df)

    # Visualizations
    print("\n🎨 GENERATING VISUALIZATIONS...")

    # Training history
    visualizer.plot_training_history(cnn_model.history, 'Enhanced CNN')

    # Confusion matrices
    predictions = {
        'Enhanced_RF': rf_predictions,
        'Enhanced_CNN': cnn_predictions
    }
    visualizer.plot_confusion_matrices(data['y_test'], predictions)

    # Model comparison
    visualizer.plot_model_comparison(comparison_df)

    # Step 5: Deployment Features
    print("\n" + "="*60)
    print("PHASE 4: DEPLOYMENT PREPARATION")
    print("="*60)

    # Create production classifier
    classifier = DigitClassifier(cnn_model, processor)

    # Test with sample predictions
    print("\n🔮 SAMPLE PREDICTIONS:")
    sample_indices = np.random.choice(len(data['X_test']), 5, replace=False)

    for i, idx in enumerate(sample_indices):
        sample_image = data['X_test'].iloc[idx].values.reshape(28, 28)
        true_label = data['y_test'].iloc[idx]

        result = classifier.predict_single_digit(sample_image)

        print(f"Sample {i+1}:")
        print(f"  True Label: {true_label}")
        print(f"  Predicted: {result['predicted_digit']}")
        print(f"  Confidence: {result['confidence']:.4f}")
        print(f"  Correct: {'✅' if result['predicted_digit'] == true_label else '❌'}")

    # Save models
    print("\n💾 SAVING MODELS...")
    classifier.save_model('enhanced_cnn_model.h5')

    with open('enhanced_rf_model.pkl', 'wb') as f:
        pickle.dump(rf_model.model, f)
    print("✅ Random Forest model saved to enhanced_rf_model.pkl")

    # Save preprocessing info
    preprocessing_info = {
        'important_pixels': processor.important_pixels.tolist(),
        'stats': processor.stats,
        'model_comparison': comparison_df.to_dict()
    }

    with open('preprocessing_info.json', 'w') as f:
        json.dump(preprocessing_info, f, indent=2)
    print("✅ Preprocessing info saved to preprocessing_info.json")

    # Step 6: Final Report
    print("\n" + "="*60)
    print("ENHANCED POC COMPLETION REPORT")
    print("="*60)

    best_model = comparison_df['accuracy'].idxmax()
    best_accuracy = comparison_df.loc[best_model, 'accuracy']
    improvement = ((best_accuracy - comparison_df['accuracy'].min()) /
                  comparison_df['accuracy'].min() * 100)

    print(f"🏆 Best Model: {best_model}")
    print(f"🎯 Best Accuracy: {best_accuracy:.4f}")
    print(f"📈 Improvement over baseline: {improvement:.2f}%")
    print(f"🔧 Features reduced by: {processor.stats['feature_reduction']:.1f}%")
    print(f"⏱️  Total training time: {sum([r.get('training_time', 0) for r in evaluator.results.values()]):.2f}s")

    print(f"\n✅ Enhanced Proof of Concept completed successfully!")
    print(f"🚀 Ready for production deployment!")
    print(f"📁 Models and artifacts saved for deployment")

    return {
        'processor': processor,
        'models': {'rf': rf_model, 'cnn': cnn_model},
        'results': comparison_df,
        'classifier': classifier
    }

# Run the enhanced PoC
if __name__ == "__main__":
    results = main()